
# 10 — Epsilon-Margin POSet Robustness  
## Final 5-Variable Country-Level Robustness Check

This notebook tests the epsilon-margin dominance rule using the continuous direction-aligned 0–100 scores from the final 5-variable structural snapshots.

It runs after:

```text
09_Epsilon_Sensitivity_Country_Level_2002_5Var.ipynb
```

## Important methodological distinction

Notebook 09 tested **epsilon as tolerance**:

```text
A_i + epsilon >= B_i
```

This notebook tests **epsilon as required margin**:

```text
A_i >= B_i for every dimension
and A is better than B by more than epsilon in at least one dimension
```

## Why this matters

The epsilon-tolerance rule can create reciprocal dominance and cycles, which may violate partial-order validity.

The epsilon-margin rule is more conservative: it only accepts dominance when one country is at least as strong in every dimension and has a minimum meaningful advantage in at least one dimension.

## Final decisions reflected here

- Final ordering variables = **5**.
- WGI/governance is not an ordering variable.
- Recovery is not an ordering variable.
- Volatility is not an ordering variable.
- Epsilon-margin is a **robustness check**, not the main POSet construction.
- Main result remains the **5-level profile POSet** from Notebook 08.

## Epsilon grid

Scores are normalized to a 0–1 scale.

The tested epsilon-margin grid is:

```text
0.00, 0.05, 0.10, 0.15, 0.20
```

`epsilon = 0.10` is retained as a report/reference robustness value because it represents a moderate 10-percentage-point minimum advantage on the normalized score scale.

## Main outputs

- `epsilon_margin_run_summary.csv`
- `epsilon_margin_country_summary.csv`
- `epsilon_margin_frontier_stability_summary.csv`
- `epsilon_margin_validity_diagnostics.csv`
- `epsilon_margin_relations.csv`
- `epsilon_tolerance_vs_margin_comparison.csv`
- `working_main_epsilon_margin_run_summary.csv`
- `report_ready_epsilon_margin_notes.csv`


In [1]:

# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import itertools
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

MASTER_DIR = PROJECT_ROOT / "Data" / "Processed" / "Master_Dataset"
EPSILON_TOLERANCE_DIR = PROJECT_ROOT / "Data" / "Processed" / "Epsilon_Sensitivity_Country_Level"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Epsilon_Margin_POSet_Robustness"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "10_Epsilon_Margin_POSet_Robustness"
TABLES_DIR = PROJECT_ROOT / "Outputs" / "Tables" / "Epsilon_Margin_POSet_Robustness"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, TABLES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Run ID:", RUN_ID)
print("Project root:", PROJECT_ROOT.resolve())
print("Master folder:", MASTER_DIR.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())


Run ID: 20260624_193325
Project root: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24
Master folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Master_Dataset
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Epsilon_Margin_POSet_Robustness


In [2]:

# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

STRUCTURAL_COMPLETE_CASE_FILE = MASTER_DIR / "structural_snapshot_final5_complete_cases.csv"
TOLERANCE_RUN_SUMMARY_FILE = EPSILON_TOLERANCE_DIR / "epsilon_run_summary.csv"

if not STRUCTURAL_COMPLETE_CASE_FILE.exists():
    raise FileNotFoundError(f"Missing structural complete-case file: {STRUCTURAL_COMPLETE_CASE_FILE}")

FINAL_5_SCORE_COLUMNS = [
    "debt_capacity_score_0_100",
    "employment_strength_score_0_100",
    "rd_intensity_score_0_100",
    "human_capital_tertiary_score_0_100",
    "energy_security_score_0_100",
]

NORMALIZED_COLUMNS = [c.replace("_score_0_100", "_score_0_1") for c in FINAL_5_SCORE_COLUMNS]

EPSILON_GRID = [0.00, 0.05, 0.10, 0.15, 0.20]

REFERENCE_EPSILON_FOR_REPORT = 0.10
REFERENCE_EPSILON_ROLE = "robustness_reference_only_not_main_profile_poset"

MAIN_POSET_NOTE = "Main result remains Notebook 08 5-level profile POSet."

print("Epsilon-margin grid:", EPSILON_GRID)
print("Reference robustness epsilon:", REFERENCE_EPSILON_FOR_REPORT)


Epsilon-margin grid: [0.0, 0.05, 0.1, 0.15, 0.2]
Reference robustness epsilon: 0.1


In [3]:

# ------------------------------------------------------
# SAVE HELPER
# ------------------------------------------------------

table_inventory_rows = []

def save_table(table, file_name, title, description):
    processed_path = PROCESSED_DIR / file_name
    diagnostics_path = DIAGNOSTICS_DIR / file_name
    table_path = TABLES_DIR / file_name

    table.to_csv(processed_path, index=False)
    table.to_csv(diagnostics_path, index=False)
    table.to_csv(table_path, index=False)

    table_inventory_rows.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "rows": len(table),
        "columns": len(table.columns),
        "processed_path": str(processed_path),
        "diagnostics_path": str(diagnostics_path),
        "table_path": str(table_path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved:", file_name)


In [4]:

# ------------------------------------------------------
# LOAD FINAL 5 COMPLETE-CASE SNAPSHOTS
# ------------------------------------------------------

data = pd.read_csv(STRUCTURAL_COMPLETE_CASE_FILE)

data["Country"] = data["Country"].astype(str).str.strip().str.upper()
data["shock_id"] = data["shock_id"].astype(str)
data["baseline_year"] = pd.to_numeric(data["baseline_year"], errors="coerce").astype(int)

missing_scores = sorted(set(FINAL_5_SCORE_COLUMNS) - set(data.columns))
if missing_scores:
    raise ValueError(f"Missing final score columns: {missing_scores}")

for col in FINAL_5_SCORE_COLUMNS:
    data[col] = pd.to_numeric(data[col], errors="coerce")
    norm_col = col.replace("_score_0_100", "_score_0_1")
    data[norm_col] = data[col] / 100.0

data = data.dropna(subset=NORMALIZED_COLUMNS).copy()

print("Loaded complete-case structural snapshot:", data.shape)
display(data.groupby(["shock_id", "baseline_year"])["Country"].nunique().reset_index(name="countries"))
display(data[["Country", "shock_id", "baseline_year"] + FINAL_5_SCORE_COLUMNS].head())


Loaded complete-case structural snapshot: (60, 51)


,shock_id,baseline_year,countries
0,2007,2007,25
1,2019,2019,35


,Country,shock_id,baseline_year,debt_capacity_score_0_100,employment_strength_score_0_100,rd_intensity_score_0_100,human_capital_tertiary_score_0_100,energy_security_score_0_100
0,AUT,2007,2007,38.5303,80.6046,68.7303,33.2757,22.6201
1,BEL,2007,2007,16.9811,46.5777,48.5360,53.5113,9.6139
2,CAN,2007,2007,64.9865,63.6697,50.3901,100.0000,100.0000
3,CZE,2007,2007,76.7627,74.6107,29.3904,0.4466,50.5562
4,DEU,2007,2007,40.6157,31.1478,68.2315,30.9850,28.5833


In [5]:

# ------------------------------------------------------
# EPSILON-MARGIN DOMINANCE FUNCTIONS
# ------------------------------------------------------

def epsilon_margin_dominates(values_a, values_b, epsilon):
    a = np.array(values_a, dtype=float)
    b = np.array(values_b, dtype=float)

    weak_condition = np.all(a >= b)
    margin_condition = np.any((a - b) > epsilon)

    return bool(weak_condition and margin_condition)


def build_epsilon_margin_relations(group, epsilon):
    records = group.to_dict("records")
    rows = []

    for a in records:
        a_values = [a[col] for col in NORMALIZED_COLUMNS]

        for b in records:
            if a["Country"] == b["Country"]:
                continue

            b_values = [b[col] for col in NORMALIZED_COLUMNS]

            if epsilon_margin_dominates(a_values, b_values, epsilon):
                rows.append({
                    "shock_id": a["shock_id"],
                    "baseline_year": a["baseline_year"],
                    "epsilon_margin": epsilon,
                    "dominating_country": a["Country"],
                    "dominated_country": b["Country"],
                })

    return pd.DataFrame(rows)


def detect_reciprocal_pairs(relations):
    if relations.empty:
        return pd.DataFrame(columns=[
            "shock_id", "baseline_year", "epsilon_margin",
            "country_a", "country_b", "cycle_type",
        ])

    pairs = set(zip(relations["dominating_country"], relations["dominated_country"]))
    reciprocal_rows = []

    for a, b in pairs:
        if (b, a) in pairs and a < b:
            first = relations.iloc[0]
            reciprocal_rows.append({
                "shock_id": first["shock_id"],
                "baseline_year": first["baseline_year"],
                "epsilon_margin": first["epsilon_margin"],
                "country_a": a,
                "country_b": b,
                "cycle_type": "reciprocal_dominance_pair",
            })

    return pd.DataFrame(reciprocal_rows)


def has_cycle(relations, countries):
    graph = {c: set() for c in countries}
    indegree = {c: 0 for c in countries}

    for _, row in relations.iterrows():
        a = row["dominating_country"]
        b = row["dominated_country"]
        if b not in graph[a]:
            graph[a].add(b)
            indegree[b] += 1

    queue = [c for c in countries if indegree[c] == 0]
    visited = 0

    while queue:
        node = queue.pop()
        visited += 1

        for child in graph[node]:
            indegree[child] -= 1
            if indegree[child] == 0:
                queue.append(child)

    return visited != len(countries)


def compute_frontier(countries, relations):
    if relations.empty:
        return sorted(countries)

    dominated = set(relations["dominated_country"].unique())
    return sorted([c for c in countries if c not in dominated])


def compute_country_counts(countries, relations):
    rows = []

    for c in countries:
        dominates_count = int((relations["dominating_country"] == c).sum()) if not relations.empty else 0
        dominated_by_count = int((relations["dominated_country"] == c).sum()) if not relations.empty else 0

        rows.append({
            "Country": c,
            "dominates_count": dominates_count,
            "dominated_by_count": dominated_by_count,
            "is_frontier": dominated_by_count == 0,
        })

    return pd.DataFrame(rows)


def compute_comparability(countries, relations):
    pairs = set()
    if not relations.empty:
        pairs = set(zip(relations["dominating_country"], relations["dominated_country"]))

    total_pairs = 0
    comparable_pairs = 0

    for a, b in itertools.combinations(sorted(countries), 2):
        total_pairs += 1
        if (a, b) in pairs or (b, a) in pairs:
            comparable_pairs += 1

    return {
        "total_country_pairs": total_pairs,
        "comparable_country_pairs": comparable_pairs,
        "incomparable_country_pairs": total_pairs - comparable_pairs,
        "comparability_ratio": comparable_pairs / total_pairs if total_pairs else np.nan,
        "incomparability_ratio": (total_pairs - comparable_pairs) / total_pairs if total_pairs else np.nan,
    }


def transitive_reduction_country(relations):
    if relations.empty:
        return relations.copy()

    pairs = set(zip(relations["dominating_country"], relations["dominated_country"]))
    direct_rows = []

    for _, row in relations.iterrows():
        a = row["dominating_country"]
        b = row["dominated_country"]

        is_transitive = False

        countries = set(relations["dominating_country"]).union(set(relations["dominated_country"]))

        for c in countries:
            if c == a or c == b:
                continue
            if (a, c) in pairs and (c, b) in pairs:
                is_transitive = True
                break

        if not is_transitive:
            direct_rows.append(row.to_dict())

    return pd.DataFrame(direct_rows)


In [6]:

# ------------------------------------------------------
# RUN EPSILON-MARGIN GRID
# ------------------------------------------------------

relation_frames = []
hasse_frames = []
country_summary_frames = []
cycle_frames = []
validity_rows = []
run_rows = []

for shock_id, group in data.groupby("shock_id"):
    group = group.copy()
    baseline_year = int(group["baseline_year"].iloc[0])
    countries = sorted(group["Country"].unique().tolist())

    for epsilon_margin in EPSILON_GRID:
        relations = build_epsilon_margin_relations(group, epsilon_margin)

        if relations.empty:
            relations = pd.DataFrame(columns=[
                "shock_id", "baseline_year", "epsilon_margin",
                "dominating_country", "dominated_country",
            ])

        hasse = transitive_reduction_country(relations)

        if hasse.empty:
            hasse = pd.DataFrame(columns=list(relations.columns) + ["edge_type"])
        else:
            hasse["edge_type"] = "epsilon_margin_hasse_cover_relation"

        reciprocal = detect_reciprocal_pairs(relations)
        cycle_exists = has_cycle(relations, countries) if not relations.empty else False
        is_valid_partial_order = not cycle_exists and reciprocal.empty

        frontier = compute_frontier(countries, relations)
        country_counts = compute_country_counts(countries, relations)
        country_counts["shock_id"] = shock_id
        country_counts["baseline_year"] = baseline_year
        country_counts["epsilon_margin"] = epsilon_margin
        country_counts["is_valid_partial_order"] = is_valid_partial_order

        comp = compute_comparability(countries, relations)

        relation_frames.append(relations)
        hasse_frames.append(hasse)
        country_summary_frames.append(country_counts)
        cycle_frames.append(reciprocal)

        validity_rows.append({
            "shock_id": shock_id,
            "baseline_year": baseline_year,
            "epsilon_margin": epsilon_margin,
            "countries": len(countries),
            "relations": len(relations),
            "hasse_edges": len(hasse),
            "reciprocal_pairs": len(reciprocal),
            "cycle_detected": cycle_exists,
            "is_valid_partial_order": is_valid_partial_order,
            "frontier_countries": len(frontier),
            "frontier_country_list": ", ".join(frontier),
            **comp,
        })

        run_rows.append({
            "shock_id": shock_id,
            "baseline_year": baseline_year,
            "epsilon_margin": epsilon_margin,
            "countries": len(countries),
            "relations": len(relations),
            "hasse_edges": len(hasse),
            "frontier_countries": len(frontier),
            "comparability_ratio": comp["comparability_ratio"],
            "incomparability_ratio": comp["incomparability_ratio"],
            "is_valid_partial_order": is_valid_partial_order,
            "robustness_role": "epsilon_margin_sensitivity_not_main_profile_poset",
        })

epsilon_margin_relations = pd.concat(relation_frames, ignore_index=True)
epsilon_margin_hasse_edges = pd.concat(hasse_frames, ignore_index=True)
epsilon_margin_country_summary = pd.concat(country_summary_frames, ignore_index=True)
epsilon_margin_cycle_diagnostics = pd.concat(cycle_frames, ignore_index=True) if cycle_frames else pd.DataFrame()
epsilon_margin_validity_diagnostics = pd.DataFrame(validity_rows)
epsilon_margin_run_summary = pd.DataFrame(run_rows)

save_table(
    epsilon_margin_relations,
    "epsilon_margin_relations.csv",
    "Epsilon-margin dominance relations",
    "Country-level epsilon-margin dominance relations for each shock and epsilon value.",
)

save_table(
    epsilon_margin_hasse_edges,
    "epsilon_margin_hasse_edges.csv",
    "Epsilon-margin Hasse edges",
    "Direct cover relations for each country-level epsilon-margin relation.",
)

save_table(
    epsilon_margin_country_summary,
    "epsilon_margin_country_summary.csv",
    "Epsilon-margin country summary",
    "Country-level domination/frontier summaries for each epsilon-margin value.",
)

save_table(
    epsilon_margin_cycle_diagnostics,
    "epsilon_margin_cycle_diagnostics.csv",
    "Epsilon-margin cycle diagnostics",
    "Reciprocal dominance/cycle diagnostics for epsilon-margin rule.",
)

save_table(
    epsilon_margin_validity_diagnostics,
    "epsilon_margin_validity_diagnostics.csv",
    "Epsilon-margin validity diagnostics",
    "Validity checks for epsilon-margin relations as partial orders.",
)

save_table(
    epsilon_margin_run_summary,
    "epsilon_margin_run_summary.csv",
    "Epsilon-margin run summary",
    "Run-level summary of country-level epsilon-margin robustness.",
)

display(epsilon_margin_run_summary)
display(epsilon_margin_validity_diagnostics)


Saved: epsilon_margin_relations.csv
Saved: epsilon_margin_hasse_edges.csv
Saved: epsilon_margin_country_summary.csv
Saved: epsilon_margin_cycle_diagnostics.csv
Saved: epsilon_margin_validity_diagnostics.csv
Saved: epsilon_margin_run_summary.csv


,shock_id,baseline_year,epsilon_margin,countries,relations,hasse_edges,frontier_countries,comparability_ratio,incomparability_ratio,is_valid_partial_order,robustness_role
0,2007,2007,0.0000,25,58,49,12,0.1933,0.8067,True,epsilon_margin_sensitivity_not_main_profile_poset
1,2007,2007,0.0500,25,58,49,12,0.1933,0.8067,True,epsilon_margin_sensitivity_not_main_profile_poset
2,2007,2007,0.1000,25,58,49,12,0.1933,0.8067,True,epsilon_margin_sensitivity_not_main_profile_poset
3,2007,2007,0.1500,25,58,49,12,0.1933,0.8067,True,epsilon_margin_sensitivity_not_main_profile_poset
4,2007,2007,0.2000,25,58,49,12,0.1933,0.8067,True,epsilon_margin_sensitivity_not_main_profile_poset
5,2019,2019,0.0000,35,81,70,20,0.1361,0.8639,True,epsilon_margin_sensitivity_not_main_profile_poset
6,2019,2019,0.0500,35,81,70,20,0.1361,0.8639,True,epsilon_margin_sensitivity_not_main_profile_poset
7,2019,2019,0.1000,35,81,70,20,0.1361,0.8639,True,epsilon_margin_sensitivity_not_main_profile_poset
8,2019,2019,0.1500,35,80,69,20,0.1345,0.8655,True,epsilon_margin_sensitivity_not_main_profile_poset
9,2019,2019,0.2000,35,76,67,20,0.1277,0.8723,True,epsilon_margin_sensitivity_not_main_profile_poset


,shock_id,baseline_year,epsilon_margin,countries,relations,hasse_edges,reciprocal_pairs,cycle_detected,is_valid_partial_order,frontier_countries,frontier_country_list,total_country_pairs,comparable_country_pairs,incomparable_country_pairs,comparability_ratio,incomparability_ratio
0,2007,2007,0.0000,25,58,49,0,False,True,12,"CAN, CZE, DNK, EST, FIN, GBR, IRL, LTU, LUX, S...",300,58,242,0.1933,0.8067
1,2007,2007,0.0500,25,58,49,0,False,True,12,"CAN, CZE, DNK, EST, FIN, GBR, IRL, LTU, LUX, S...",300,58,242,0.1933,0.8067
2,2007,2007,0.1000,25,58,49,0,False,True,12,"CAN, CZE, DNK, EST, FIN, GBR, IRL, LTU, LUX, S...",300,58,242,0.1933,0.8067
3,2007,2007,0.1500,25,58,49,0,False,True,12,"CAN, CZE, DNK, EST, FIN, GBR, IRL, LTU, LUX, S...",300,58,242,0.1933,0.8067
4,2007,2007,0.2000,25,58,49,0,False,True,12,"CAN, CZE, DNK, EST, FIN, GBR, IRL, LTU, LUX, S...",300,58,242,0.1933,0.8067
5,2019,2019,0.0000,35,81,70,0,False,True,20,"AUS, AUT, BGR, CAN, CHE, CZE, DEU, DNK, EST, F...",595,81,514,0.1361,0.8639
6,2019,2019,0.0500,35,81,70,0,False,True,20,"AUS, AUT, BGR, CAN, CHE, CZE, DEU, DNK, EST, F...",595,81,514,0.1361,0.8639
7,2019,2019,0.1000,35,81,70,0,False,True,20,"AUS, AUT, BGR, CAN, CHE, CZE, DEU, DNK, EST, F...",595,81,514,0.1361,0.8639
8,2019,2019,0.1500,35,80,69,0,False,True,20,"AUS, AUT, BGR, CAN, CHE, CZE, DEU, DNK, EST, F...",595,80,515,0.1345,0.8655
9,2019,2019,0.2000,35,76,67,0,False,True,20,"AUS, AUT, BGR, CAN, CHE, CZE, DEU, DNK, EST, F...",595,76,519,0.1277,0.8723


In [7]:

# ------------------------------------------------------
# FRONTIER STABILITY SUMMARY
# ------------------------------------------------------

frontier_rows = []

for shock_id, group in epsilon_margin_country_summary.groupby("shock_id"):
    baseline_frontier = set(
        group[
            group["epsilon_margin"].eq(0.00)
            & group["is_frontier"]
        ]["Country"]
    )

    for epsilon_margin, eps_group in group.groupby("epsilon_margin"):
        frontier = set(eps_group[eps_group["is_frontier"]]["Country"])

        intersection = baseline_frontier.intersection(frontier)
        union = baseline_frontier.union(frontier)

        frontier_rows.append({
            "shock_id": shock_id,
            "epsilon_margin": epsilon_margin,
            "baseline_epsilon_margin": 0.00,
            "frontier_countries": len(frontier),
            "frontier_country_list": ", ".join(sorted(frontier)),
            "baseline_frontier_countries": len(baseline_frontier),
            "countries_shared_with_margin_0": len(intersection),
            "jaccard_similarity_with_margin_0": len(intersection) / len(union) if union else np.nan,
            "countries_added_vs_margin_0": ", ".join(sorted(frontier - baseline_frontier)),
            "countries_removed_vs_margin_0": ", ".join(sorted(baseline_frontier - frontier)),
        })

epsilon_margin_frontier_stability_summary = pd.DataFrame(frontier_rows)

working_main_epsilon_margin_run_summary = epsilon_margin_run_summary[
    epsilon_margin_run_summary["epsilon_margin"].eq(REFERENCE_EPSILON_FOR_REPORT)
].copy()
working_main_epsilon_margin_run_summary["reference_role"] = REFERENCE_EPSILON_ROLE

working_main_epsilon_margin_frontier_stability_summary = epsilon_margin_frontier_stability_summary[
    epsilon_margin_frontier_stability_summary["epsilon_margin"].eq(REFERENCE_EPSILON_FOR_REPORT)
].copy()
working_main_epsilon_margin_frontier_stability_summary["reference_role"] = REFERENCE_EPSILON_ROLE

save_table(
    epsilon_margin_frontier_stability_summary,
    "epsilon_margin_frontier_stability_summary.csv",
    "Epsilon-margin frontier stability summary",
    "Frontier stability compared with epsilon-margin = 0.00 baseline.",
)

save_table(
    working_main_epsilon_margin_run_summary,
    "working_main_epsilon_margin_run_summary.csv",
    "Working main epsilon-margin run summary",
    "Reference epsilon-margin robustness summary. Not the main profile POSet result.",
)

save_table(
    working_main_epsilon_margin_frontier_stability_summary,
    "working_main_epsilon_margin_frontier_stability_summary.csv",
    "Working main epsilon-margin frontier stability summary",
    "Frontier stability at reference epsilon-margin. Not the main profile POSet result.",
)

display(epsilon_margin_frontier_stability_summary)
display(working_main_epsilon_margin_run_summary)


Saved: epsilon_margin_frontier_stability_summary.csv
Saved: working_main_epsilon_margin_run_summary.csv
Saved: working_main_epsilon_margin_frontier_stability_summary.csv


,shock_id,epsilon_margin,baseline_epsilon_margin,frontier_countries,frontier_country_list,baseline_frontier_countries,countries_shared_with_margin_0,jaccard_similarity_with_margin_0,countries_added_vs_margin_0,countries_removed_vs_margin_0
0,2007,0.0000,0.0000,12,"CAN, CZE, DNK, EST, FIN, GBR, IRL, LTU, LUX, S...",12,12,1.0000,,
1,2007,0.0500,0.0000,12,"CAN, CZE, DNK, EST, FIN, GBR, IRL, LTU, LUX, S...",12,12,1.0000,,
2,2007,0.1000,0.0000,12,"CAN, CZE, DNK, EST, FIN, GBR, IRL, LTU, LUX, S...",12,12,1.0000,,
3,2007,0.1500,0.0000,12,"CAN, CZE, DNK, EST, FIN, GBR, IRL, LTU, LUX, S...",12,12,1.0000,,
4,2007,0.2000,0.0000,12,"CAN, CZE, DNK, EST, FIN, GBR, IRL, LTU, LUX, S...",12,12,1.0000,,
5,2019,0.0000,0.0000,20,"AUS, AUT, BGR, CAN, CHE, CZE, DEU, DNK, EST, F...",20,20,1.0000,,
6,2019,0.0500,0.0000,20,"AUS, AUT, BGR, CAN, CHE, CZE, DEU, DNK, EST, F...",20,20,1.0000,,
7,2019,0.1000,0.0000,20,"AUS, AUT, BGR, CAN, CHE, CZE, DEU, DNK, EST, F...",20,20,1.0000,,
8,2019,0.1500,0.0000,20,"AUS, AUT, BGR, CAN, CHE, CZE, DEU, DNK, EST, F...",20,20,1.0000,,
9,2019,0.2000,0.0000,20,"AUS, AUT, BGR, CAN, CHE, CZE, DEU, DNK, EST, F...",20,20,1.0000,,


,shock_id,baseline_year,epsilon_margin,countries,relations,hasse_edges,frontier_countries,comparability_ratio,incomparability_ratio,is_valid_partial_order,robustness_role,reference_role
2,2007,2007,0.1000,25,58,49,12,0.1933,0.8067,True,epsilon_margin_sensitivity_not_main_profile_poset,robustness_reference_only_not_main_profile_poset
7,2019,2019,0.1000,35,81,70,20,0.1361,0.8639,True,epsilon_margin_sensitivity_not_main_profile_poset,robustness_reference_only_not_main_profile_poset


In [8]:

# ------------------------------------------------------
# COMPARE EPSILON TOLERANCE VS EPSILON MARGIN
# ------------------------------------------------------

if TOLERANCE_RUN_SUMMARY_FILE.exists():
    tolerance = pd.read_csv(TOLERANCE_RUN_SUMMARY_FILE)
    tolerance["shock_id"] = tolerance["shock_id"].astype(str)
    tolerance = tolerance.rename(columns={"epsilon": "epsilon_value"})
    tolerance["rule"] = "epsilon_tolerance"

    margin = epsilon_margin_run_summary.rename(columns={"epsilon_margin": "epsilon_value"}).copy()
    margin["rule"] = "epsilon_margin"

    common_cols = [
        "shock_id", "baseline_year", "epsilon_value", "countries",
        "relations", "frontier_countries", "comparability_ratio",
        "incomparability_ratio", "is_valid_partial_order", "rule",
    ]

    tolerance_vs_margin_long = pd.concat(
        [tolerance[common_cols], margin[common_cols]],
        ignore_index=True,
    )

    wide = tolerance_vs_margin_long.pivot_table(
        index=["shock_id", "baseline_year", "epsilon_value"],
        columns="rule",
        values=["relations", "frontier_countries", "comparability_ratio", "is_valid_partial_order"],
        aggfunc="first",
    )

    wide.columns = ["_".join(map(str, col)).strip() for col in wide.columns.values]
    epsilon_tolerance_vs_margin_comparison = wide.reset_index()

else:
    tolerance_vs_margin_long = pd.DataFrame()
    epsilon_tolerance_vs_margin_comparison = pd.DataFrame([
        {
            "note": "Tolerance run summary from Notebook 09 not found. Comparison skipped.",
        }
    ])

save_table(
    tolerance_vs_margin_long,
    "epsilon_tolerance_vs_margin_long.csv",
    "Epsilon tolerance vs margin long",
    "Long-format comparison between epsilon-tolerance and epsilon-margin rules.",
)

save_table(
    epsilon_tolerance_vs_margin_comparison,
    "epsilon_tolerance_vs_margin_comparison.csv",
    "Epsilon tolerance vs margin comparison",
    "Side-by-side comparison between epsilon-tolerance and epsilon-margin rules.",
)

display(epsilon_tolerance_vs_margin_comparison)


Saved: epsilon_tolerance_vs_margin_long.csv
Saved: epsilon_tolerance_vs_margin_comparison.csv


,shock_id,baseline_year,epsilon_value,comparability_ratio_epsilon_margin,comparability_ratio_epsilon_tolerance,frontier_countries_epsilon_margin,frontier_countries_epsilon_tolerance,is_valid_partial_order_epsilon_margin,is_valid_partial_order_epsilon_tolerance,relations_epsilon_margin,relations_epsilon_tolerance
0,2007,2007,0.0000,0.1933,0.1933,12,12,True,True,58,58
1,2007,2007,0.0500,0.1933,0.2600,12,10,True,True,58,78
2,2007,2007,0.1000,0.1933,0.3500,12,7,True,True,58,105
3,2007,2007,0.1500,0.1933,0.4467,12,4,True,False,58,136
4,2007,2007,0.2000,0.1933,0.5567,12,4,True,False,58,171
5,2019,2019,0.0000,0.1361,0.1361,20,20,True,True,81,81
6,2019,2019,0.0500,0.1361,0.2723,20,9,True,True,81,162
7,2019,2019,0.1000,0.1361,0.4134,20,5,True,False,81,248
8,2019,2019,0.1500,0.1345,0.5647,20,3,True,False,80,347
9,2019,2019,0.2000,0.1277,0.6840,20,3,True,False,76,442


In [9]:

# ------------------------------------------------------
# REPORT-READY EPSILON-MARGIN NOTES
# ------------------------------------------------------

report_ready_epsilon_margin_notes = pd.DataFrame([
    {
        "topic": "Epsilon-margin rule",
        "report_text": (
            "The epsilon-margin robustness rule requires country A to be at least as strong as country B in all "
            "five normalized dimensions and better by more than epsilon in at least one dimension."
        ),
    },
    {
        "topic": "Epsilon value and grid",
        "report_text": (
            "The robustness grid tests epsilon-margin values of 0.00, 0.05, 0.10, 0.15, and 0.20 on the "
            "0–1 normalized score scale. Epsilon = 0.10 is retained as a moderate reference value, equivalent "
            "to requiring at least one 10-percentage-point advantage."
        ),
    },
    {
        "topic": "Why epsilon-margin",
        "report_text": (
            "Compared with epsilon-tolerance, epsilon-margin is more conservative because it does not relax "
            "the requirement that A must be at least as good as B in every dimension. It only raises the minimum "
            "advantage needed to record dominance."
        ),
    },
    {
        "topic": "Relationship to main POSet",
        "report_text": (
            "The epsilon-margin analysis is a robustness check. The main empirical structure remains the "
            "five-level profile POSet, while epsilon-margin is used to examine how sensitive dominance and "
            "frontier membership are to requiring a minimum advantage."
        ),
    },
    {
        "topic": "Validity check",
        "report_text": (
            "Each epsilon-margin relation is checked for reciprocal dominance and cycles. Unlike tolerance-based "
            "dominance, the margin rule should remain a valid partial order because it is a stricter subset of "
            "standard componentwise dominance."
        ),
    },
])

methodological_decision_updates_epsilon_margin = pd.DataFrame([
    {
        "decision": "Test epsilon-margin robustness",
        "choice": "epsilon-margin grid 0.00 to 0.20",
        "reason": "Assesses whether dominance/frontier results depend on tiny differences in normalized scores.",
        "risk_controlled": "Makes the epsilon-margin value and sensitivity analysis explicit.",
    },
    {
        "decision": "Use epsilon = 0.10 as reference robustness value",
        "choice": "0.10 on 0–1 normalized scale",
        "reason": "Moderate threshold requiring at least one 10-percentage-point advantage.",
        "risk_controlled": "Avoids arbitrary unstated epsilon choice.",
    },
    {
        "decision": "Keep epsilon-margin outside main result",
        "choice": "robustness/sensitivity only",
        "reason": "Main report structure is based on the interpretable 5-level profile POSet.",
        "risk_controlled": "Avoids replacing the main ordinal POSet with multiple competing country-level relations.",
    },
])

save_table(
    report_ready_epsilon_margin_notes,
    "report_ready_epsilon_margin_notes.csv",
    "Report-ready epsilon-margin notes",
    "Report-ready notes describing epsilon-margin rule, value, grid, and interpretation.",
)

save_table(
    methodological_decision_updates_epsilon_margin,
    "methodological_decision_updates_epsilon_margin.csv",
    "Methodological decision updates epsilon margin",
    "Decision-log entries for epsilon-margin robustness.",
)

display(report_ready_epsilon_margin_notes)
display(methodological_decision_updates_epsilon_margin)


Saved: report_ready_epsilon_margin_notes.csv
Saved: methodological_decision_updates_epsilon_margin.csv


,topic,report_text
0,Epsilon-margin rule,The epsilon-margin robustness rule requires co...
1,Epsilon value and grid,The robustness grid tests epsilon-margin value...
2,Why epsilon-margin,"Compared with epsilon-tolerance, epsilon-margi..."
3,Relationship to main POSet,The epsilon-margin analysis is a robustness ch...
4,Validity check,Each epsilon-margin relation is checked for re...


,decision,choice,reason,risk_controlled
0,Test epsilon-margin robustness,epsilon-margin grid 0.00 to 0.20,Assesses whether dominance/frontier results de...,Makes the epsilon-margin value and sensitivity...
1,Use epsilon = 0.10 as reference robustness value,0.10 on 0–1 normalized scale,Moderate threshold requiring at least one 10-p...,Avoids arbitrary unstated epsilon choice.
2,Keep epsilon-margin outside main result,robustness/sensitivity only,Main report structure is based on the interpre...,Avoids replacing the main ordinal POSet with m...


In [10]:

# ------------------------------------------------------
# AUDIT WORKBOOK
# ------------------------------------------------------

audit_xlsx = AUDIT_DIR / "10_Epsilon_Margin_POSet_Robustness_Audit.xlsx"

with pd.ExcelWriter(audit_xlsx, engine="xlsxwriter") as writer:
    epsilon_margin_run_summary.to_excel(writer, sheet_name="run_summary", index=False)
    epsilon_margin_validity_diagnostics.to_excel(writer, sheet_name="validity", index=False)
    epsilon_margin_cycle_diagnostics.to_excel(writer, sheet_name="cycles", index=False)
    epsilon_margin_frontier_stability_summary.to_excel(writer, sheet_name="frontier_stability", index=False)
    working_main_epsilon_margin_run_summary.to_excel(writer, sheet_name="reference_margin_run", index=False)
    working_main_epsilon_margin_frontier_stability_summary.to_excel(writer, sheet_name="reference_margin_frontier", index=False)
    epsilon_tolerance_vs_margin_comparison.to_excel(writer, sheet_name="tolerance_vs_margin", index=False)
    epsilon_margin_country_summary.to_excel(writer, sheet_name="country_summary", index=False)
    report_ready_epsilon_margin_notes.to_excel(writer, sheet_name="report_notes", index=False)
    methodological_decision_updates_epsilon_margin.to_excel(writer, sheet_name="decision_updates", index=False)
    pd.DataFrame(table_inventory_rows).to_excel(writer, sheet_name="table_inventory", index=False)

print("Audit workbook saved:", audit_xlsx.resolve())


Audit workbook saved: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Audit\10_Epsilon_Margin_POSet_Robustness_Audit.xlsx


In [11]:

# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

table_inventory = pd.DataFrame(table_inventory_rows)
table_inventory.to_csv(PROCESSED_DIR / "epsilon_margin_table_inventory.csv", index=False)
table_inventory.to_csv(DIAGNOSTICS_DIR / "epsilon_margin_table_inventory.csv", index=False)

expected_outputs = [
    "epsilon_margin_relations.csv",
    "epsilon_margin_hasse_edges.csv",
    "epsilon_margin_country_summary.csv",
    "epsilon_margin_cycle_diagnostics.csv",
    "epsilon_margin_validity_diagnostics.csv",
    "epsilon_margin_run_summary.csv",
    "epsilon_margin_frontier_stability_summary.csv",
    "working_main_epsilon_margin_run_summary.csv",
    "working_main_epsilon_margin_frontier_stability_summary.csv",
    "epsilon_tolerance_vs_margin_comparison.csv",
    "epsilon_tolerance_vs_margin_long.csv",
    "report_ready_epsilon_margin_notes.csv",
    "methodological_decision_updates_epsilon_margin.csv",
]

output_check = pd.DataFrame([
    {
        "file_name": f,
        "processed_exists": (PROCESSED_DIR / f).exists(),
        "diagnostics_exists": (DIAGNOSTICS_DIR / f).exists(),
        "tables_exists": (TABLES_DIR / f).exists(),
    }
    for f in expected_outputs
])

output_check.to_csv(DIAGNOSTICS_DIR / "output_check.csv", index=False)

print("10 EPSILON-MARGIN POSET ROBUSTNESS COMPLETE")
print("=" * 90)

display(output_check)

print("\nKey outputs to inspect/send:")
print("- 10_Epsilon_Margin_POSet_Robustness_Audit.xlsx")
print("- epsilon_margin_run_summary.csv")
print("- epsilon_margin_validity_diagnostics.csv")
print("- epsilon_margin_frontier_stability_summary.csv")
print("- epsilon_tolerance_vs_margin_comparison.csv")

print("\nNext notebook:")
print("11_Recovery_Validation_2002_5Var.ipynb")


10 EPSILON-MARGIN POSET ROBUSTNESS COMPLETE


,file_name,processed_exists,diagnostics_exists,tables_exists
0,epsilon_margin_relations.csv,True,True,True
1,epsilon_margin_hasse_edges.csv,True,True,True
2,epsilon_margin_country_summary.csv,True,True,True
3,epsilon_margin_cycle_diagnostics.csv,True,True,True
4,epsilon_margin_validity_diagnostics.csv,True,True,True
5,epsilon_margin_run_summary.csv,True,True,True
6,epsilon_margin_frontier_stability_summary.csv,True,True,True
7,working_main_epsilon_margin_run_summary.csv,True,True,True
8,working_main_epsilon_margin_frontier_stability...,True,True,True
9,epsilon_tolerance_vs_margin_comparison.csv,True,True,True



Key outputs to inspect/send:
- 10_Epsilon_Margin_POSet_Robustness_Audit.xlsx
- epsilon_margin_run_summary.csv
- epsilon_margin_validity_diagnostics.csv
- epsilon_margin_frontier_stability_summary.csv
- epsilon_tolerance_vs_margin_comparison.csv

Next notebook:
11_Recovery_Validation_2002_5Var.ipynb
